# Multi-Model Pose Comparison for Horse Riding Analysis

**Completely self-contained notebook** - just run all cells!

This notebook compares multiple pose estimation models on horse riding videos with **real-time biomechanics metrics**:

### Core Metrics Tracked (Real-Time)
| Metric | Description |
|--------|-------------|
| **Lower Leg Stability** | Measures leg swing relative to hip |
| **Rein Symmetry** | Whether hands are carried at even heights |
| **Rein Steadiness** | Hand movement relative to shoulders |
| **Upper Body Vertical Alignment** | Trunk lean from vertical |
| **Core Stability** | Torso stability relative to pelvis |
| **Pelvis Vertical Stability** | Smooth following of horse motion |

### Outputs
1. Per-model annotated videos with real-time metrics overlay
2. Sprite video (2x3 grid) showing models side-by-side
3. CSV with aggregate metrics for comparison

---
## 1. Install Dependencies

In [ ]:
# Install core dependencies
!pip install -q ultralytics>=8.0.0 opencv-python>=4.8.0 numpy>=1.24.0 matplotlib>=3.7.0 scipy>=1.10.0 pandas

# Install MediaPipe (optional but recommended)
!pip install -q mediapipe>=0.10.0

# Note: RTMPose/MMPose requires more complex installation
# Uncomment below if you want RTMPose support:
# !pip install -q openmim
# !mim install mmengine mmcv mmdet mmpose

print("\n" + "="*50)
print("Installation complete!")
print("="*50)

---
## 2. Imports

In [ ]:
import sys
import time
import csv
import shutil
import subprocess
from pathlib import Path
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple
from enum import Enum
from collections import deque

import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Video, display, HTML

# Check for optional dependencies
try:
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
    print("✓ Ultralytics YOLO available")
except ImportError:
    YOLO_AVAILABLE = False
    print("✗ Ultralytics not available")

try:
    import mediapipe as mp
    MEDIAPIPE_AVAILABLE = True
    print("✓ MediaPipe available")
except ImportError:
    MEDIAPIPE_AVAILABLE = False
    print("✗ MediaPipe not available")

try:
    from mmpose.apis import MMPoseInferencer
    RTMPOSE_AVAILABLE = True
    print("✓ RTMPose/MMPose available")
except ImportError:
    RTMPOSE_AVAILABLE = False
    print("✗ RTMPose not available (optional)")

print("\n✓ All imports successful")

---
## 3. Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - Modify these settings as needed
# =============================================================================

# Directories
VIDEOS_DIR = Path("videos")  # Input videos directory
OUTPUT_DIR = Path("comparison_output")  # Output directory

# Models to compare
MODELS_TO_USE = [
    "yolov8n-pose",  # Nano - fastest
    "yolov8s-pose",  # Small
    "yolov8m-pose",  # Medium
]

if MEDIAPIPE_AVAILABLE:
    MODELS_TO_USE.append("mediapipe-full")

if RTMPOSE_AVAILABLE:
    MODELS_TO_USE.append("rtmpose-medium")

# Sprite video settings
SPRITE_COLS = 3
SPRITE_ROWS = 2
CELL_WIDTH = 640
CELL_HEIGHT = 360

# Biomechanics settings
SLIDING_WINDOW_SIZE = 30  # Frames for real-time metric computation
CONFIDENCE_THRESHOLD = 0.5  # Minimum keypoint confidence

# Create output directory
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Videos directory: {VIDEOS_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Models to compare: {MODELS_TO_USE}")
print(f"Sliding window: {SLIDING_WINDOW_SIZE} frames")

---
## 4. Core Data Structures

In [ ]:
# =============================================================================
# KEYPOINT DEFINITIONS (COCO 17-keypoint format)
# =============================================================================

class Keypoint(Enum):
    """COCO 17 keypoint indices."""
    NOSE = 0
    LEFT_EYE = 1
    RIGHT_EYE = 2
    LEFT_EAR = 3
    RIGHT_EAR = 4
    LEFT_SHOULDER = 5
    RIGHT_SHOULDER = 6
    LEFT_ELBOW = 7
    RIGHT_ELBOW = 8
    LEFT_WRIST = 9
    RIGHT_WRIST = 10
    LEFT_HIP = 11
    RIGHT_HIP = 12
    LEFT_KNEE = 13
    RIGHT_KNEE = 14
    LEFT_ANKLE = 15
    RIGHT_ANKLE = 16


# Skeleton connections for visualization
SKELETON_CONNECTIONS = [
    (0, 1), (0, 2), (1, 3), (2, 4),  # Face
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),  # Arms
    (5, 11), (6, 12), (11, 12),  # Torso
    (11, 13), (13, 15), (12, 14), (14, 16),  # Legs
]

# MediaPipe to COCO keypoint mapping
MEDIAPIPE_TO_COCO = {
    0: 0, 2: 1, 5: 2, 7: 3, 8: 4,
    11: 5, 12: 6, 13: 7, 14: 8, 15: 9, 16: 10,
    23: 11, 24: 12, 25: 13, 26: 14, 27: 15, 28: 16,
}


@dataclass
class PoseResult:
    """Result from a single pose detection."""
    keypoints: np.ndarray  # Shape: (17, 2)
    confidences: np.ndarray  # Shape: (17,)
    inference_time_ms: float
    detected: bool = True

    @classmethod
    def empty(cls, inference_time_ms: float = 0.0) -> "PoseResult":
        return cls(
            keypoints=np.zeros((17, 2)),
            confidences=np.zeros(17),
            inference_time_ms=inference_time_ms,
            detected=False
        )

print("✓ Core data structures defined")

---
## 5. Biomechanics Metrics (Real-Time Computation)

In [ ]:
# =============================================================================
# BIOMECHANICS METRIC FUNCTIONS
# These compute metrics over a sliding window of frames for real-time display
# =============================================================================

def _rms(values: np.ndarray) -> float:
    """Compute root mean square."""
    return float(np.sqrt(np.mean(values ** 2)))


def _angle_from_vertical(vector: np.ndarray) -> float:
    """Compute angle of vector from vertical (y-axis pointing down in image coords)."""
    vertical = np.array([0, -1])  # Up direction
    cos_angle = np.dot(vector, vertical) / (np.linalg.norm(vector) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0))))


@dataclass
class BiomechanicsMetrics:
    """Real-time biomechanics metrics computed over sliding window."""
    lower_leg_stability: Optional[float] = None
    rein_symmetry: Optional[float] = None
    rein_steadiness: Optional[float] = None
    upper_body_vertical_alignment: Optional[float] = None
    core_stability: Optional[float] = None
    pelvis_vertical_stability: Optional[float] = None
    
    # Ratings for display
    lower_leg_stability_rating: str = ""
    rein_symmetry_rating: str = ""
    rein_steadiness_rating: str = ""
    upper_body_alignment_rating: str = ""
    core_stability_rating: str = ""
    pelvis_stability_rating: str = ""


class BiomechanicsComputer:
    """
    Computes biomechanics metrics over a sliding window of frames.
    
    Tracks the 6 core metrics:
    1. Lower Leg Stability
    2. Rein Symmetry
    3. Rein Steadiness
    4. Upper Body Vertical Alignment
    5. Core Stability
    6. Pelvis Vertical Stability
    """
    
    def __init__(self, window_size: int = 30, conf_threshold: float = 0.5):
        self.window_size = window_size
        self.conf_threshold = conf_threshold
        
        # Sliding window buffers
        self.keypoints_buffer: deque = deque(maxlen=window_size)
        self.confidences_buffer: deque = deque(maxlen=window_size)
    
    def reset(self):
        """Reset the sliding window buffers."""
        self.keypoints_buffer.clear()
        self.confidences_buffer.clear()
    
    def add_frame(self, keypoints: np.ndarray, confidences: np.ndarray):
        """Add a frame to the sliding window."""
        self.keypoints_buffer.append(keypoints.copy())
        self.confidences_buffer.append(confidences.copy())
    
    def compute_metrics(self) -> BiomechanicsMetrics:
        """Compute all 6 core metrics over the current sliding window."""
        metrics = BiomechanicsMetrics()
        
        if len(self.keypoints_buffer) < 5:
            return metrics
        
        # Convert to numpy arrays
        kp_array = np.array(list(self.keypoints_buffer))  # (T, 17, 2)
        conf_array = np.array(list(self.confidences_buffer))  # (T, 17)
        
        # Compute each metric
        metrics.lower_leg_stability, metrics.lower_leg_stability_rating = \
            self._compute_lower_leg_stability(kp_array, conf_array)
        
        metrics.rein_symmetry, metrics.rein_symmetry_rating = \
            self._compute_rein_symmetry(kp_array, conf_array)
        
        metrics.rein_steadiness, metrics.rein_steadiness_rating = \
            self._compute_rein_steadiness(kp_array, conf_array)
        
        metrics.upper_body_vertical_alignment, metrics.upper_body_alignment_rating = \
            self._compute_upper_body_alignment(kp_array, conf_array)
        
        metrics.core_stability, metrics.core_stability_rating = \
            self._compute_core_stability(kp_array, conf_array)
        
        metrics.pelvis_vertical_stability, metrics.pelvis_stability_rating = \
            self._compute_pelvis_stability(kp_array, conf_array)
        
        return metrics
    
    def _get_valid_mask(self, conf_array: np.ndarray, *keypoint_indices) -> np.ndarray:
        """Get mask of frames where all specified keypoints are valid."""
        mask = np.ones(len(conf_array), dtype=bool)
        for kp_idx in keypoint_indices:
            mask &= conf_array[:, kp_idx] >= self.conf_threshold
        return mask
    
    def _compute_lower_leg_stability(self, kp: np.ndarray, conf: np.ndarray) -> Tuple[Optional[float], str]:
        """
        Lower Leg Stability: Measures how much the lower leg swings relative to the hip.
        Formula: RMS(ankle_position_relative_to_hip - mean) / torso_length
        Lower is better. Target: < 0.03 excellent, < 0.06 good, < 0.10 fair
        """
        valid = self._get_valid_mask(
            conf,
            Keypoint.LEFT_HIP.value, Keypoint.RIGHT_HIP.value,
            Keypoint.LEFT_ANKLE.value, Keypoint.RIGHT_ANKLE.value,
            Keypoint.LEFT_SHOULDER.value, Keypoint.RIGHT_SHOULDER.value
        )
        
        if valid.sum() < 5:
            return None, ""
        
        # Get keypoints
        hip_mid = (kp[valid, Keypoint.LEFT_HIP.value] + kp[valid, Keypoint.RIGHT_HIP.value]) / 2
        ankle_mid = (kp[valid, Keypoint.LEFT_ANKLE.value] + kp[valid, Keypoint.RIGHT_ANKLE.value]) / 2
        shoulder_mid = (kp[valid, Keypoint.LEFT_SHOULDER.value] + kp[valid, Keypoint.RIGHT_SHOULDER.value]) / 2
        
        # Torso length for normalization
        torso_lengths = np.linalg.norm(shoulder_mid - hip_mid, axis=1)
        torso_length = np.mean(torso_lengths)
        if torso_length < 1:
            return None, ""
        
        # Leg position relative to hip
        v_leg = ankle_mid - hip_mid
        v_leg_mean = np.mean(v_leg, axis=0)
        deviations = np.linalg.norm(v_leg - v_leg_mean, axis=1)
        stability = _rms(deviations) / torso_length
        
        # Rating
        if stability < 0.03:
            rating = "excellent"
        elif stability < 0.06:
            rating = "good"
        elif stability < 0.10:
            rating = "fair"
        else:
            rating = "poor"
        
        return round(stability, 4), rating
    
    def _compute_rein_symmetry(self, kp: np.ndarray, conf: np.ndarray) -> Tuple[Optional[float], str]:
        """
        Rein Symmetry: Measures whether hands are carried at even heights.
        Formula: |y(left_wrist) - y(right_wrist)| / shoulder_width
        Lower is better. Target: < 0.02 excellent, < 0.05 good, < 0.08 fair
        """
        valid = self._get_valid_mask(
            conf,
            Keypoint.LEFT_WRIST.value, Keypoint.RIGHT_WRIST.value,
            Keypoint.LEFT_SHOULDER.value, Keypoint.RIGHT_SHOULDER.value
        )
        
        if valid.sum() < 5:
            return None, ""
        
        left_wrist = kp[valid, Keypoint.LEFT_WRIST.value]
        right_wrist = kp[valid, Keypoint.RIGHT_WRIST.value]
        left_shoulder = kp[valid, Keypoint.LEFT_SHOULDER.value]
        right_shoulder = kp[valid, Keypoint.RIGHT_SHOULDER.value]
        
        # Mean positions
        lw_mean = np.mean(left_wrist, axis=0)
        rw_mean = np.mean(right_wrist, axis=0)
        ls_mean = np.mean(left_shoulder, axis=0)
        rs_mean = np.mean(right_shoulder, axis=0)
        
        shoulder_width = np.linalg.norm(ls_mean - rs_mean)
        if shoulder_width < 1:
            return None, ""
        
        symmetry = abs(lw_mean[1] - rw_mean[1]) / shoulder_width
        
        if symmetry < 0.02:
            rating = "excellent"
        elif symmetry < 0.05:
            rating = "good"
        elif symmetry < 0.08:
            rating = "fair"
        else:
            rating = "poor"
        
        return round(symmetry, 4), rating
    
    def _compute_rein_steadiness(self, kp: np.ndarray, conf: np.ndarray) -> Tuple[Optional[float], str]:
        """
        Rein Steadiness: Measures hand movement relative to shoulders.
        Formula: RMS(hand_offset_from_shoulder - mean) / torso_length
        Lower is better. Target: < 0.02 excellent, < 0.05 good, < 0.08 fair
        """
        valid = self._get_valid_mask(
            conf,
            Keypoint.LEFT_WRIST.value, Keypoint.RIGHT_WRIST.value,
            Keypoint.LEFT_SHOULDER.value, Keypoint.RIGHT_SHOULDER.value,
            Keypoint.LEFT_HIP.value, Keypoint.RIGHT_HIP.value
        )
        
        if valid.sum() < 5:
            return None, ""
        
        # Torso length
        shoulder_mid = (kp[valid, Keypoint.LEFT_SHOULDER.value] + kp[valid, Keypoint.RIGHT_SHOULDER.value]) / 2
        hip_mid = (kp[valid, Keypoint.LEFT_HIP.value] + kp[valid, Keypoint.RIGHT_HIP.value]) / 2
        torso_length = np.mean(np.linalg.norm(shoulder_mid - hip_mid, axis=1))
        if torso_length < 1:
            return None, ""
        
        # Hand offsets from shoulders
        left_offset = kp[valid, Keypoint.LEFT_WRIST.value] - kp[valid, Keypoint.LEFT_SHOULDER.value]
        right_offset = kp[valid, Keypoint.RIGHT_WRIST.value] - kp[valid, Keypoint.RIGHT_SHOULDER.value]
        hand_offsets = (left_offset + right_offset) / 2
        
        mean_offset = np.mean(hand_offsets, axis=0)
        deviations = np.linalg.norm(hand_offsets - mean_offset, axis=1)
        steadiness = _rms(deviations) / torso_length
        
        if steadiness < 0.02:
            rating = "excellent"
        elif steadiness < 0.05:
            rating = "good"
        elif steadiness < 0.08:
            rating = "fair"
        else:
            rating = "poor"
        
        return round(steadiness, 4), rating
    
    def _compute_upper_body_alignment(self, kp: np.ndarray, conf: np.ndarray) -> Tuple[Optional[float], str]:
        """
        Upper Body Vertical Alignment: Measures trunk lean from vertical.
        Formula: mean(angle(trunk_vector, vertical))
        Lower is better. Target: < 5° excellent, < 10° good, < 15° fair
        """
        valid = self._get_valid_mask(
            conf,
            Keypoint.LEFT_SHOULDER.value, Keypoint.RIGHT_SHOULDER.value,
            Keypoint.LEFT_HIP.value, Keypoint.RIGHT_HIP.value
        )
        
        if valid.sum() < 5:
            return None, ""
        
        shoulder_mid = (kp[valid, Keypoint.LEFT_SHOULDER.value] + kp[valid, Keypoint.RIGHT_SHOULDER.value]) / 2
        hip_mid = (kp[valid, Keypoint.LEFT_HIP.value] + kp[valid, Keypoint.RIGHT_HIP.value]) / 2
        
        angles = []
        for i in range(len(shoulder_mid)):
            trunk_vec = shoulder_mid[i] - hip_mid[i]
            angles.append(_angle_from_vertical(trunk_vec))
        
        alignment = np.mean(angles)
        
        if alignment < 5:
            rating = "excellent"
        elif alignment < 10:
            rating = "good"
        elif alignment < 15:
            rating = "fair"
        else:
            rating = "poor"
        
        return round(alignment, 2), rating
    
    def _compute_core_stability(self, kp: np.ndarray, conf: np.ndarray) -> Tuple[Optional[float], str]:
        """
        Core Stability: Measures torso stability relative to pelvis.
        Formula: RMS(trunk_vector - mean) / torso_length
        Lower is better. Target: < 0.015 excellent, < 0.035 good, < 0.07 fair
        """
        valid = self._get_valid_mask(
            conf,
            Keypoint.LEFT_SHOULDER.value, Keypoint.RIGHT_SHOULDER.value,
            Keypoint.LEFT_HIP.value, Keypoint.RIGHT_HIP.value
        )
        
        if valid.sum() < 5:
            return None, ""
        
        shoulder_mid = (kp[valid, Keypoint.LEFT_SHOULDER.value] + kp[valid, Keypoint.RIGHT_SHOULDER.value]) / 2
        hip_mid = (kp[valid, Keypoint.LEFT_HIP.value] + kp[valid, Keypoint.RIGHT_HIP.value]) / 2
        
        trunk_vectors = shoulder_mid - hip_mid
        torso_length = np.mean(np.linalg.norm(trunk_vectors, axis=1))
        if torso_length < 1:
            return None, ""
        
        trunk_mean = np.mean(trunk_vectors, axis=0)
        deviations = np.linalg.norm(trunk_vectors - trunk_mean, axis=1)
        stability = _rms(deviations) / torso_length
        
        if stability < 0.015:
            rating = "excellent"
        elif stability < 0.035:
            rating = "good"
        elif stability < 0.07:
            rating = "fair"
        else:
            rating = "poor"
        
        return round(stability, 4), rating
    
    def _compute_pelvis_stability(self, kp: np.ndarray, conf: np.ndarray) -> Tuple[Optional[float], str]:
        """
        Pelvis Vertical Stability: Measures smooth following of horse motion.
        Formula: StdDev(hip_y) / torso_length
        Ideal range: 0.01-0.03 (not zero - pelvis should follow horse)
        """
        valid = self._get_valid_mask(
            conf,
            Keypoint.LEFT_HIP.value, Keypoint.RIGHT_HIP.value,
            Keypoint.LEFT_SHOULDER.value, Keypoint.RIGHT_SHOULDER.value
        )
        
        if valid.sum() < 5:
            return None, ""
        
        shoulder_mid = (kp[valid, Keypoint.LEFT_SHOULDER.value] + kp[valid, Keypoint.RIGHT_SHOULDER.value]) / 2
        hip_mid = (kp[valid, Keypoint.LEFT_HIP.value] + kp[valid, Keypoint.RIGHT_HIP.value]) / 2
        
        torso_length = np.mean(np.linalg.norm(shoulder_mid - hip_mid, axis=1))
        if torso_length < 1:
            return None, ""
        
        hip_y = hip_mid[:, 1]
        stability = np.std(hip_y) / torso_length
        
        # Special rating - ideal is 0.01-0.03, not zero
        if 0.01 <= stability <= 0.03:
            rating = "excellent"
        elif 0.03 < stability <= 0.05:
            rating = "good"
        elif 0.05 < stability <= 0.08:
            rating = "fair"
        elif stability < 0.01:
            rating = "rigid"  # Too stiff, not following horse
        else:
            rating = "poor"
        
        return round(stability, 4), rating

print("✓ BiomechanicsComputer defined with 6 core metrics")

---
## 6. Pose Model Implementations

In [ ]:
# =============================================================================
# BASE POSE MODEL
# =============================================================================

class BasePoseModel(ABC):
    """Abstract base class for pose estimation models."""

    def __init__(self, model_name: str, variant: str):
        self.model_name = model_name
        self.variant = variant
        self._model = None
        self._detection_model = None
        self._horse_class_id = 17
        self._last_horse_bbox = None
        self._horse_missing_frames = 0

    @property
    def full_name(self) -> str:
        return f"{self.model_name}_{self.variant}"

    @abstractmethod
    def load_model(self) -> None:
        pass

    @abstractmethod
    def detect_pose(self, frame: np.ndarray) -> PoseResult:
        pass

    def detect_rider_pose(self, frame: np.ndarray, horse_bbox: Optional[np.ndarray] = None) -> PoseResult:
        if horse_bbox is None:
            horse_bbox = self._find_horse_bbox(frame)
        result = self.detect_pose(frame)
        if horse_bbox is None or not result.detected:
            return result
        if self._is_rider_pose(result.keypoints, result.confidences, horse_bbox):
            return result
        return PoseResult.empty(result.inference_time_ms)

    def _find_horse_bbox(self, frame: np.ndarray) -> Optional[np.ndarray]:
        if self._detection_model is None:
            return self._last_horse_bbox
        results = self._detection_model(frame, verbose=False, classes=[self._horse_class_id], conf=0.25)
        if results and results[0].boxes is not None and len(results[0].boxes) > 0:
            boxes = results[0].boxes.data.cpu().numpy()
            largest_idx = ((boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])).argmax()
            self._last_horse_bbox = boxes[largest_idx][:4]
            self._horse_missing_frames = 0
            return self._last_horse_bbox
        if self._last_horse_bbox is not None and self._horse_missing_frames < 10:
            self._horse_missing_frames += 1
            return self._last_horse_bbox
        return None

    def _is_rider_pose(self, kp: np.ndarray, conf: np.ndarray, bbox: np.ndarray) -> bool:
        if conf[Keypoint.LEFT_HIP.value] < 0.4 or conf[Keypoint.RIGHT_HIP.value] < 0.4:
            return False
        hip_mid = (kp[Keypoint.LEFT_HIP.value] + kp[Keypoint.RIGHT_HIP.value]) / 2
        x1, y1, x2, y2 = bbox
        margin = (x2 - x1) * 0.3
        if hip_mid[0] < x1 - margin or hip_mid[0] > x2 + margin:
            return False
        if hip_mid[1] > (y1 + y2) / 2:
            return False
        return True

    def reset_cache(self):
        self._last_horse_bbox = None
        self._horse_missing_frames = 0

print("✓ BasePoseModel defined")

In [ ]:
# =============================================================================
# YOLO POSE MODEL
# =============================================================================

class YOLOPoseModel(BasePoseModel):
    YOLOV8_VARIANTS = ["yolov8n-pose", "yolov8s-pose", "yolov8m-pose", "yolov8l-pose", "yolov8x-pose"]
    YOLOV11_VARIANTS = ["yolo11n-pose", "yolo11s-pose", "yolo11m-pose"]

    def __init__(self, variant: str = "yolov8n-pose"):
        name = "YOLO-Pose" if variant in self.YOLOV8_VARIANTS else "YOLOv11-Pose"
        super().__init__(name, variant)

    def load_model(self):
        from ultralytics import YOLO
        self._model = YOLO(f"{self.variant}.pt")
        self._detection_model = YOLO("yolov8n.pt")

    def detect_pose(self, frame: np.ndarray) -> PoseResult:
        start = time.perf_counter()
        results = self._model(frame, verbose=False)
        elapsed = (time.perf_counter() - start) * 1000
        if not results or results[0].keypoints is None or len(results[0].keypoints) == 0:
            return PoseResult.empty(elapsed)
        kp = results[0].keypoints.xy.cpu().numpy()
        conf = results[0].keypoints.conf.cpu().numpy()
        best = conf.mean(axis=1).argmax()
        return PoseResult(kp[best], conf[best], elapsed, True)

    def detect_rider_pose(self, frame: np.ndarray, horse_bbox: Optional[np.ndarray] = None) -> PoseResult:
        start = time.perf_counter()
        if horse_bbox is None:
            horse_bbox = self._find_horse_bbox(frame)
        results = self._model(frame, verbose=False)
        elapsed = (time.perf_counter() - start) * 1000
        if not results or results[0].keypoints is None or len(results[0].keypoints) == 0:
            return PoseResult.empty(elapsed)
        kp = results[0].keypoints.xy.cpu().numpy()
        conf = results[0].keypoints.conf.cpu().numpy()
        if horse_bbox is None:
            best = conf.mean(axis=1).argmax()
            return PoseResult(kp[best], conf[best], elapsed, True)
        # Find rider
        x1, y1, x2, y2 = horse_bbox
        horse_cx, horse_cy = (x1 + x2) / 2, (y1 + y2) / 2
        best_idx, best_dist = -1, float('inf')
        for i in range(len(kp)):
            if conf[i][Keypoint.LEFT_HIP.value] < 0.4 or conf[i][Keypoint.RIGHT_HIP.value] < 0.4:
                continue
            hip = (kp[i][Keypoint.LEFT_HIP.value] + kp[i][Keypoint.RIGHT_HIP.value]) / 2
            if hip[0] < x1 - (x2-x1)*0.3 or hip[0] > x2 + (x2-x1)*0.3 or hip[1] > horse_cy:
                continue
            dist = abs(hip[0] - horse_cx)
            if dist < best_dist:
                best_idx, best_dist = i, dist
        if best_idx < 0:
            return PoseResult.empty(elapsed)
        return PoseResult(kp[best_idx], conf[best_idx], elapsed, True)

print("✓ YOLOPoseModel defined")

In [ ]:
# =============================================================================
# MEDIAPIPE POSE MODEL
# =============================================================================

class MediaPipePoseModel(BasePoseModel):
    VARIANTS = ["lite", "full", "heavy"]
    COMPLEXITY = {"lite": 0, "full": 1, "heavy": 2}

    def __init__(self, variant: str = "full"):
        super().__init__("MediaPipe", variant)
        self._pose = None

    def load_model(self):
        import mediapipe as mp
        from ultralytics import YOLO
        self._pose = mp.solutions.pose.Pose(
            static_image_mode=False,
            model_complexity=self.COMPLEXITY[self.variant],
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        self._detection_model = YOLO("yolov8n.pt")

    def detect_pose(self, frame: np.ndarray) -> PoseResult:
        start = time.perf_counter()
        results = self._pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        elapsed = (time.perf_counter() - start) * 1000
        if results.pose_landmarks is None:
            return PoseResult.empty(elapsed)
        h, w = frame.shape[:2]
        kp, conf = np.zeros((17, 2)), np.zeros(17)
        for mp_idx, coco_idx in MEDIAPIPE_TO_COCO.items():
            lm = results.pose_landmarks.landmark[mp_idx]
            kp[coco_idx] = [lm.x * w, lm.y * h]
            conf[coco_idx] = lm.visibility
        return PoseResult(kp, conf, elapsed, True)

    def __del__(self):
        if self._pose:
            self._pose.close()

print("✓ MediaPipePoseModel defined")

In [ ]:
# =============================================================================
# RTMPOSE MODEL (Optional)
# =============================================================================

class RTMPoseModel(BasePoseModel):
    VARIANTS = ["tiny", "small", "medium", "large"]
    CONFIGS = {
        "tiny": "rtmpose-t_8xb256-420e_coco-256x192",
        "small": "rtmpose-s_8xb256-420e_coco-256x192",
        "medium": "rtmpose-m_8xb256-420e_coco-256x192",
        "large": "rtmpose-l_8xb256-420e_coco-256x192",
    }

    def __init__(self, variant: str = "medium"):
        super().__init__("RTMPose", variant)
        self._inferencer = None

    def load_model(self):
        from mmpose.apis import MMPoseInferencer
        from ultralytics import YOLO
        import torch
        device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
        self._inferencer = MMPoseInferencer(pose2d=self.CONFIGS[self.variant], device=device)
        self._detection_model = YOLO("yolov8n.pt")

    def detect_pose(self, frame: np.ndarray) -> PoseResult:
        start = time.perf_counter()
        results = list(self._inferencer(frame, return_vis=False))
        elapsed = (time.perf_counter() - start) * 1000
        if not results or 'predictions' not in results[0] or not results[0]['predictions'][0]:
            return PoseResult.empty(elapsed)
        preds = results[0]['predictions'][0]
        best = max(preds, key=lambda p: np.mean(p.get('keypoint_scores', [0])))
        return PoseResult(np.array(best['keypoints'])[:, :2], np.array(best['keypoint_scores']), elapsed, True)

print("✓ RTMPoseModel defined")

---
## 7. Real-Time Metrics Overlay

In [ ]:
# =============================================================================
# METRICS OVERLAY WITH 6 CORE BIOMECHANICS METRICS
# =============================================================================

class MetricsOverlay:
    """Draws real-time biomechanics metrics overlay on video frames."""
    
    # Rating colors (BGR)
    COLORS = {
        "excellent": (0, 200, 0),    # Green
        "good": (0, 200, 200),       # Yellow
        "fair": (0, 165, 255),       # Orange
        "poor": (0, 0, 255),         # Red
        "rigid": (255, 0, 255),      # Magenta
        "": (200, 200, 200),         # Gray (no data)
    }

    def __init__(self, font_scale: float = 0.45, line_height: int = 18, padding: int = 8):
        self.font = cv2.FONT_HERSHEY_SIMPLEX
        self.font_scale = font_scale
        self.line_height = line_height
        self.padding = padding

    def draw_overlay(self, frame: np.ndarray, model_name: str, model_variant: str,
                     metrics: BiomechanicsMetrics, inference_time: float,
                     detection_rate: float, frame_idx: int, total_frames: int) -> np.ndarray:
        """Draw complete overlay with model info and biomechanics metrics."""
        # Top-left: Model info
        self._draw_model_info(frame, model_name, model_variant, inference_time)
        
        # Top-right: Core biomechanics metrics (the 6 key metrics)
        self._draw_biomechanics_panel(frame, metrics)
        
        # Bottom-left: Progress info
        self._draw_progress_info(frame, frame_idx, total_frames, detection_rate)
        
        return frame

    def _draw_box(self, frame: np.ndarray, lines: List[Tuple[str, Tuple[int,int,int]]], 
                  x: int, y: int, title: str = None) -> int:
        """Draw a semi-transparent box with colored text lines. Returns box height."""
        # Calculate dimensions
        max_width = 0
        for text, _ in lines:
            (w, _), _ = cv2.getTextSize(text, self.font, self.font_scale, 1)
            max_width = max(max_width, w)
        if title:
            (tw, _), _ = cv2.getTextSize(title, self.font, self.font_scale, 1)
            max_width = max(max_width, tw)
        
        title_lines = 1 if title else 0
        box_h = (len(lines) + title_lines) * self.line_height + 2 * self.padding
        box_w = max_width + 2 * self.padding
        
        # Draw semi-transparent background
        overlay = frame.copy()
        cv2.rectangle(overlay, (x, y), (x + box_w, y + box_h), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.7, frame, 0.3, 0, frame)
        cv2.rectangle(frame, (x, y), (x + box_w, y + box_h), (255, 255, 255), 1)
        
        # Draw title
        text_y = y + self.padding + int(self.line_height * 0.7)
        if title:
            cv2.putText(frame, title, (x + self.padding, text_y),
                       self.font, self.font_scale, (0, 255, 255), 1, cv2.LINE_AA)
            text_y += self.line_height
        
        # Draw lines with colors
        for text, color in lines:
            cv2.putText(frame, text, (x + self.padding, text_y),
                       self.font, self.font_scale, color, 1, cv2.LINE_AA)
            text_y += self.line_height
        
        return box_h

    def _draw_model_info(self, frame: np.ndarray, model_name: str, variant: str, inference_time: float):
        """Draw model info in top-left corner."""
        lines = [
            (f"Model: {model_name}", (255, 255, 255)),
            (f"Variant: {variant}", (255, 255, 255)),
            (f"Inference: {inference_time:.1f}ms", (200, 200, 200)),
        ]
        self._draw_box(frame, lines, self.padding, self.padding)

    def _draw_biomechanics_panel(self, frame: np.ndarray, metrics: BiomechanicsMetrics):
        """Draw the 6 core biomechanics metrics in top-right corner."""
        h, w = frame.shape[:2]
        
        # Format each metric with value and rating
        def fmt(name: str, value: Optional[float], rating: str, unit: str = "") -> Tuple[str, Tuple]:
            if value is None:
                return f"{name}: --", self.COLORS[""]
            val_str = f"{value:.3f}" if unit == "" else f"{value:.1f}{unit}"
            return f"{name}: {val_str} [{rating}]", self.COLORS.get(rating, self.COLORS[""])
        
        lines = [
            fmt("Lower Leg", metrics.lower_leg_stability, metrics.lower_leg_stability_rating),
            fmt("Rein Sym", metrics.rein_symmetry, metrics.rein_symmetry_rating),
            fmt("Rein Steady", metrics.rein_steadiness, metrics.rein_steadiness_rating),
            fmt("Upper Body", metrics.upper_body_vertical_alignment, metrics.upper_body_alignment_rating, "°"),
            fmt("Core", metrics.core_stability, metrics.core_stability_rating),
            fmt("Pelvis", metrics.pelvis_vertical_stability, metrics.pelvis_stability_rating),
        ]
        
        # Calculate position for top-right
        max_width = 220  # Approximate width
        x = w - max_width - self.padding
        
        self._draw_box(frame, lines, x, self.padding, "BIOMECHANICS")

    def _draw_progress_info(self, frame: np.ndarray, frame_idx: int, total_frames: int, detection_rate: float):
        """Draw progress info in bottom-left corner."""
        h, w = frame.shape[:2]
        
        lines = [
            (f"Frame: {frame_idx + 1}/{total_frames}", (255, 255, 255)),
            (f"Detection: {detection_rate:.1%}", (200, 200, 200)),
        ]
        
        box_h = len(lines) * self.line_height + 2 * self.padding
        y = h - box_h - self.padding
        
        self._draw_box(frame, lines, self.padding, y)

    def draw_model_label(self, frame: np.ndarray, model_name: str, variant: str) -> np.ndarray:
        """Draw simple model label for sprite video."""
        label = f"{model_name}: {variant}"
        (tw, th), _ = cv2.getTextSize(label, self.font, self.font_scale, 1)
        h, w = frame.shape[:2]
        x, y = (w - tw) // 2, self.padding + th
        cv2.rectangle(frame, (x - 5, y - th - 5), (x + tw + 5, y + 5), (0, 0, 0), -1)
        cv2.putText(frame, label, (x, y), self.font, self.font_scale, (0, 255, 255), 1, cv2.LINE_AA)
        return frame

print("✓ MetricsOverlay defined with 6 core biomechanics metrics")

---
## 8. Video Processor

In [ ]:
# =============================================================================
# VIDEO METRICS DATA STRUCTURES
# =============================================================================

@dataclass
class FrameMetrics:
    """Metrics for a single frame."""
    inference_time_ms: float = 0.0
    detected: bool = False
    biomechanics: Optional[BiomechanicsMetrics] = None


@dataclass 
class VideoMetrics:
    """Aggregated metrics for an entire video."""
    model_name: str
    model_variant: str
    video_file: str
    frame_metrics: List[FrameMetrics] = field(default_factory=list)
    total_frames: int = 0
    fps: float = 0.0

    @property
    def detection_rate(self) -> float:
        if not self.frame_metrics:
            return 0.0
        return sum(1 for fm in self.frame_metrics if fm.detected) / len(self.frame_metrics)

    @property
    def avg_inference_time_ms(self) -> float:
        if not self.frame_metrics:
            return 0.0
        return np.mean([fm.inference_time_ms for fm in self.frame_metrics])

print("✓ Video metrics data structures defined")

In [ ]:
# =============================================================================
# MULTI-MODEL VIDEO PROCESSOR
# =============================================================================

class MultiModelVideoProcessor:
    """Process videos with multiple pose models and real-time biomechanics."""

    def __init__(self, models: List[BasePoseModel], output_dir: str, 
                 window_size: int = 30):
        self.models = models
        self.output_dir = Path(output_dir)
        self.window_size = window_size
        self.overlay = MetricsOverlay()

    def process_video(self, video_path: str, verbose: bool = True) -> Dict[str, VideoMetrics]:
        """Process video with all models, generating annotated videos with biomechanics."""
        video_path = Path(video_path)
        video_stem = video_path.stem
        video_output_dir = self.output_dir / video_stem
        video_output_dir.mkdir(parents=True, exist_ok=True)

        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        if verbose:
            print(f"  Video: {width}x{height}, {total_frames} frames, {fps:.1f} fps")

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        
        # Initialize per-model resources
        writers = {}
        all_metrics = {}
        bio_computers = {}

        for model in self.models:
            out_path = video_output_dir / f"{model.full_name}_annotated.mp4"
            writers[model.full_name] = cv2.VideoWriter(str(out_path), fourcc, fps, (width, height))
            all_metrics[model.full_name] = VideoMetrics(
                model_name=model.model_name, model_variant=model.variant,
                video_file=str(video_path), total_frames=total_frames, fps=fps
            )
            bio_computers[model.full_name] = BiomechanicsComputer(window_size=self.window_size)
            model.reset_cache()

        frame_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # Detect horse once per frame (shared)
            horse_bbox = self.models[0]._find_horse_bbox(frame) if self.models else None

            for model in self.models:
                name = model.full_name
                
                # Detect pose
                result = model.detect_rider_pose(frame, horse_bbox)
                
                # Update biomechanics computer
                if result.detected:
                    bio_computers[name].add_frame(result.keypoints, result.confidences)
                
                # Compute current metrics
                bio_metrics = bio_computers[name].compute_metrics()
                
                # Store frame metrics
                fm = FrameMetrics(
                    inference_time_ms=result.inference_time_ms,
                    detected=result.detected,
                    biomechanics=bio_metrics
                )
                all_metrics[name].frame_metrics.append(fm)
                
                # Draw annotated frame
                annotated = frame.copy()
                if result.detected:
                    annotated = self._draw_skeleton(annotated, result.keypoints, result.confidences)
                
                # Current detection rate
                det_count = sum(1 for f in all_metrics[name].frame_metrics if f.detected)
                det_rate = det_count / len(all_metrics[name].frame_metrics)
                
                # Draw overlay
                annotated = self.overlay.draw_overlay(
                    annotated, model.model_name, model.variant,
                    bio_metrics, result.inference_time_ms,
                    det_rate, frame_idx, total_frames
                )
                
                writers[name].write(annotated)

            frame_idx += 1
            if verbose and frame_idx % 100 == 0:
                print(f"    Processed {frame_idx}/{total_frames} frames...")

        cap.release()
        for w in writers.values():
            w.release()

        if verbose:
            print(f"  ✓ Output saved to: {video_output_dir}")
            for model in self.models:
                m = all_metrics[model.full_name]
                print(f"    {model.full_name}: det={m.detection_rate:.1%}, time={m.avg_inference_time_ms:.1f}ms")

        return all_metrics

    def _draw_skeleton(self, frame: np.ndarray, kp: np.ndarray, conf: np.ndarray) -> np.ndarray:
        """Draw pose skeleton."""
        for i, j in SKELETON_CONNECTIONS:
            if conf[i] > 0.5 and conf[j] > 0.5:
                cv2.line(frame, tuple(kp[i].astype(int)), tuple(kp[j].astype(int)), (255, 255, 0), 2)
        for i, (pt, c) in enumerate(zip(kp, conf)):
            if c > 0.5:
                cv2.circle(frame, tuple(pt.astype(int)), 4, (0, 255, 0), -1)
        return frame

print("✓ MultiModelVideoProcessor defined with real-time biomechanics")

---
## 9. Sprite Composer & Metrics Aggregator

In [ ]:
# =============================================================================
# SPRITE COMPOSER
# =============================================================================

class SpriteComposer:
    """Compose sprite videos showing multiple models in a grid."""

    def __init__(self, cell_width: int = 640, cell_height: int = 360, 
                 grid_cols: int = 3, grid_rows: int = 2):
        self.cell_width = cell_width
        self.cell_height = cell_height
        self.grid_cols = grid_cols
        self.grid_rows = grid_rows
        self.total_width = cell_width * grid_cols
        self.total_height = cell_height * grid_rows
        self.overlay = MetricsOverlay()

    def compose(self, video_paths: Dict[str, str], output_path: str,
                model_order: Optional[List[str]] = None, verbose: bool = True) -> str:
        """Compose sprite video from annotated videos."""
        ordered = [video_paths[n] for n in (model_order or list(video_paths.keys())) if n in video_paths]
        max_cells = self.grid_rows * self.grid_cols
        
        caps = [(Path(p).stem.replace("_annotated", ""), cv2.VideoCapture(p)) for p in ordered[:max_cells]]
        caps = [(n, c) for n, c in caps if c.isOpened()]
        
        if not caps:
            raise ValueError("No videos could be opened")
        
        fps = caps[0][1].get(cv2.CAP_PROP_FPS)
        total = int(caps[0][1].get(cv2.CAP_PROP_FRAME_COUNT))
        
        out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps,
                              (self.total_width, self.total_height))
        
        if verbose:
            print("  Composing sprite video...")
        
        frame_idx = 0
        while True:
            frames, done = [], True
            for name, cap in caps:
                ret, f = cap.read()
                if ret:
                    done = False
                    f = cv2.resize(f, (self.cell_width, self.cell_height))
                else:
                    f = np.zeros((self.cell_height, self.cell_width, 3), dtype=np.uint8)
                frames.append((name, f))
            
            if done:
                break
            
            grid = np.zeros((self.total_height, self.total_width, 3), dtype=np.uint8)
            for i, (name, f) in enumerate(frames[:max_cells]):
                row, col = i // self.grid_cols, i % self.grid_cols
                y1, x1 = row * self.cell_height, col * self.cell_width
                parts = name.split("_")
                f = self.overlay.draw_model_label(f, parts[0], "_".join(parts[1:]))
                grid[y1:y1+self.cell_height, x1:x1+self.cell_width] = f
            
            out.write(grid)
            frame_idx += 1
            if verbose and frame_idx % 100 == 0:
                print(f"    Composed {frame_idx}/{total} frames...")
        
        for _, c in caps:
            c.release()
        out.release()
        
        if verbose:
            print(f"  ✓ Sprite saved: {output_path}")
        return output_path

print("✓ SpriteComposer defined")

In [ ]:
# =============================================================================
# METRICS AGGREGATOR
# =============================================================================

class MetricsAggregator:
    """Aggregates metrics and exports to CSV."""

    def __init__(self):
        self.results = []

    def add_video_metrics(self, vm: VideoMetrics):
        """Add video metrics and compute averages."""
        detected = [fm for fm in vm.frame_metrics if fm.detected and fm.biomechanics]
        
        row = {
            "model_name": vm.model_name,
            "model_variant": vm.model_variant,
            "video_file": vm.video_file,
            "detection_rate": vm.detection_rate,
            "avg_inference_time_ms": vm.avg_inference_time_ms,
            "lower_leg_stability": None,
            "rein_symmetry": None,
            "rein_steadiness": None,
            "upper_body_alignment": None,
            "core_stability": None,
            "pelvis_stability": None,
        }
        
        if detected:
            def avg(attr):
                vals = [getattr(fm.biomechanics, attr) for fm in detected if getattr(fm.biomechanics, attr) is not None]
                return np.mean(vals) if vals else None
            
            row["lower_leg_stability"] = avg("lower_leg_stability")
            row["rein_symmetry"] = avg("rein_symmetry")
            row["rein_steadiness"] = avg("rein_steadiness")
            row["upper_body_alignment"] = avg("upper_body_vertical_alignment")
            row["core_stability"] = avg("core_stability")
            row["pelvis_stability"] = avg("pelvis_vertical_stability")
        
        self.results.append(row)

    def export_csv(self, output_path: str) -> str:
        """Export to CSV."""
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        df = pd.DataFrame(self.results)
        df.to_csv(output_path, index=False)
        return output_path

    def get_summary(self) -> Dict:
        if not self.results:
            return {}
        df = pd.DataFrame(self.results)
        summary = {"total_videos": len(df["video_file"].unique()), "models": {}}
        for name, grp in df.groupby(["model_name", "model_variant"]):
            key = f"{name[0]}_{name[1]}"
            summary["models"][key] = {
                "detection_rate": grp["detection_rate"].mean(),
                "inference_time_ms": grp["avg_inference_time_ms"].mean(),
            }
        return summary

print("✓ MetricsAggregator defined")

---
## 10. Helper Functions

In [ ]:
def create_model(variant: str) -> Optional[BasePoseModel]:
    """Create a pose model from variant string."""
    variant = variant.lower().strip()
    if variant.startswith("yolov8") or variant.startswith("yolo11"):
        return YOLOPoseModel(variant=variant)
    if variant.startswith("mediapipe-"):
        return MediaPipePoseModel(variant=variant.replace("mediapipe-", "")) if MEDIAPIPE_AVAILABLE else None
    if variant.startswith("rtmpose-"):
        return RTMPoseModel(variant=variant.replace("rtmpose-", "")) if RTMPOSE_AVAILABLE else None
    return None


def find_videos(videos_dir: Path) -> List[Path]:
    """Find all video files."""
    if not videos_dir.exists():
        return []
    exts = [".mp4", ".avi", ".mov", ".mkv"]
    videos = []
    for e in exts:
        videos.extend(videos_dir.glob(f"*{e}"))
        videos.extend(videos_dir.glob(f"*{e.upper()}"))
    return sorted(set(videos))


def get_video_info(path: Path) -> Dict:
    """Get video info."""
    cap = cv2.VideoCapture(str(path))
    info = {
        "frames": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        "fps": cap.get(cv2.CAP_PROP_FPS),
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    info["duration"] = info["frames"] / max(info["fps"], 1)
    cap.release()
    return info

print("✓ Helper functions defined")

---
## 11. Find Videos & Load Models

In [ ]:
# Find videos
videos = find_videos(VIDEOS_DIR)
print(f"Found {len(videos)} videos:")
for v in videos:
    info = get_video_info(v)
    print(f"  • {v.name}: {info['width']}x{info['height']}, {info['duration']:.1f}s")

if not videos:
    print(f"\n⚠ No videos found in '{VIDEOS_DIR}'")

In [ ]:
# Load models
print("\nLoading models...")
models = []
for variant in MODELS_TO_USE:
    print(f"  {variant}...", end=" ", flush=True)
    try:
        model = create_model(variant)
        if model:
            model.load_model()
            models.append(model)
            print("✓")
        else:
            print("skipped")
    except Exception as e:
        print(f"✗ ({e})")

print(f"\n✓ Loaded {len(models)} models")

---
## 12. Process Videos with Real-Time Biomechanics

In [ ]:
# Process videos
if videos and models:
    processor = MultiModelVideoProcessor(
        models=models, 
        output_dir=str(OUTPUT_DIR),
        window_size=SLIDING_WINDOW_SIZE
    )
    aggregator = MetricsAggregator()
    all_metrics = {}

    for video_path in videos:
        print(f"\n{'='*60}")
        print(f"Processing: {video_path.name}")
        print(f"{'='*60}")
        
        try:
            vm = processor.process_video(str(video_path), verbose=True)
            all_metrics[video_path.stem] = vm
            for model in models:
                aggregator.add_video_metrics(vm[model.full_name])
        except Exception as e:
            print(f"Error: {e}")
            import traceback
            traceback.print_exc()

    print(f"\n✓ Processed {len(all_metrics)} videos with real-time biomechanics")
else:
    print("⚠ No videos or models available")

---
## 13. Generate Sprite Videos

In [ ]:
# Generate sprites
sprite_videos = []

if videos and models:
    composer = SpriteComposer(CELL_WIDTH, CELL_HEIGHT, SPRITE_COLS, SPRITE_ROWS)

    for video_path in videos:
        stem = video_path.stem
        out_dir = OUTPUT_DIR / stem
        
        print(f"\nGenerating sprite for: {stem}")
        
        paths = {m.full_name: str(out_dir / f"{m.full_name}_annotated.mp4") 
                 for m in models if (out_dir / f"{m.full_name}_annotated.mp4").exists()}
        
        if len(paths) < 2:
            print("  ⚠ Not enough videos")
            continue
        
        sprite_path = out_dir / f"{stem}_sprite.mp4"
        try:
            composer.compose(paths, str(sprite_path), [m.full_name for m in models], verbose=True)
            sprite_videos.append(sprite_path)
        except Exception as e:
            print(f"  ✗ {e}")

    print(f"\n✓ Generated {len(sprite_videos)} sprite videos")

---
## 14. Export Results & Visualize

In [ ]:
# Export CSV
if 'aggregator' in dir() and aggregator.results:
    csv_path = OUTPUT_DIR / "model_comparison.csv"
    aggregator.export_csv(str(csv_path))
    print(f"✓ Exported: {csv_path}")
    
    df = pd.read_csv(csv_path)
    display(df)

In [ ]:
# Visualize biomechanics metrics
if 'df' in dir() and len(df) > 0:
    metrics_cols = ["lower_leg_stability", "rein_symmetry", "rein_steadiness",
                    "upper_body_alignment", "core_stability", "pelvis_stability"]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    model_labels = df['model_name'] + '\n' + df['model_variant']
    x = range(len(df))
    
    titles = ["Lower Leg Stability", "Rein Symmetry", "Rein Steadiness",
              "Upper Body Alignment (°)", "Core Stability", "Pelvis Stability"]
    colors = ['steelblue', 'coral', 'seagreen', 'purple', 'orange', 'teal']
    
    for i, (col, title, color) in enumerate(zip(metrics_cols, titles, colors)):
        vals = df[col].fillna(0)
        axes[i].bar(x, vals, color=color)
        axes[i].set_xticks(x)
        axes[i].set_xticklabels(model_labels, rotation=45, ha='right')
        axes[i].set_title(title)
        axes[i].set_ylabel('Value (lower=better)' if 'alignment' not in col else 'Degrees')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'biomechanics_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n✓ Saved: {OUTPUT_DIR / 'biomechanics_comparison.png'}")

---
## 15. Display Sprite Video

In [ ]:
if sprite_videos:
    sprite = sprite_videos[0]
    info = get_video_info(sprite)
    print(f"Sprite: {sprite.name}")
    print(f"Size: {info['width']}x{info['height']}, Duration: {info['duration']:.1f}s")
    display(Video(str(sprite), width=960, embed=True))
else:
    print("No sprite videos")

---
## 16. Summary

In [ ]:
print("="*60)
print("COMPARISON SUMMARY")
print("="*60)

if 'aggregator' in dir() and aggregator.results:
    summary = aggregator.get_summary()
    print(f"\nVideos: {summary.get('total_videos', 0)}")
    print(f"Models: {len(summary.get('models', {}))}")
    
    print("\nPer-model results:")
    for key, stats in summary.get('models', {}).items():
        print(f"  {key}:")
        print(f"    Detection: {stats['detection_rate']:.1%}")
        print(f"    Inference: {stats['inference_time_ms']:.1f}ms")

print("\n" + "="*60)
print("OUTPUT FILES")
print("="*60)
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(OUTPUT_DIR)} ({f.stat().st_size/1024/1024:.1f} MB)")

print("\n✓ COMPLETE!")

---

## Done!

The annotated videos now show **real-time biomechanics metrics**:

| Metric | What It Measures | Good Range |
|--------|------------------|------------|
| **Lower Leg Stability** | Leg swing relative to hip | < 0.06 |
| **Rein Symmetry** | Hand height evenness | < 0.05 |
| **Rein Steadiness** | Hand movement stability | < 0.05 |
| **Upper Body Alignment** | Trunk lean from vertical | < 10° |
| **Core Stability** | Torso stability | < 0.035 |
| **Pelvis Stability** | Following horse motion | 0.01-0.03 |

Check `comparison_output/` for all results!